# Train SLM Router - Qwen3-1.7B Multi-task Fine-tuning (Kaggle)

**Task:** Fine-tune Qwen3-1.7B for dual-output routing:
- Primary: Intent classification + scope detection (15 intents x 3 scopes)
- Secondary (multi-task): Contextual query rewriting with conversation context

**Output schema:**
```json
// Turn 1: {"intent": "current_weather", "scope": "district", "confidence": 0.95}
// Turn 2+: {"intent": "daily_forecast", "scope": "district", "confidence": 0.91,
//           "rewritten_query": "Du bao ngay mai quan Cau Giay?"}
```

## Setup (bat buoc truoc khi chay)
1. Chay `python scripts/router/generate_rewrite_data.py --use-llm` de tao `multitask_train.jsonl` va `multitask_val.jsonl`
2. Upload ca 2 file len Kaggle Datasets
2. Settings -> Add Data -> chon dataset vua upload
3. Settings -> Accelerator -> GPU T4 x2 hoac P100
4. Settings -> Internet -> On
5. Settings -> Secrets -> them `HF_TOKEN` (optional, neu muon push Hub)


In [ ]:
# Cell 1: Install packages
# unsloth[kaggle-new] = build toi uu cho Kaggle T4/P100
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "unsloth[kaggle-new]", "trl>=0.11,<0.16",
], check=True)
print("Packages installed")

In [ ]:
# Cell 2: Imports
import json, os
from pathlib import Path
from collections import Counter
import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 3: Configuration

# --- Model ---
BASE_MODEL = "unsloth/Qwen3-1.7B-bnb-4bit"
OUTPUT_DIR = "/kaggle/working/outputs"
GGUF_DIR   = "/kaggle/working/gguf"

# --- Data ---
# Thay DATASET_NAME bang slug cua Kaggle dataset ban da upload
DATASET_NAME   = "hanoi-weather-router"  # <- doi thanh ten dataset cua ban
DATA_DIR       = Path(f"/kaggle/input/{DATASET_NAME}")
TRAIN_FILE     = DATA_DIR / "multitask_train.jsonl"  # unified: routing + rewrite
VAL_FILE       = DATA_DIR / "multitask_val.jsonl"    # converted val set

# --- LoRA ---
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.0

# --- Training ---
MAX_SEQ_LENGTH = 1024   # Tang tu 768 cho truong rewritten_query
EPOCHS         = 5
BATCH_SIZE     = 4      # T4 16GB handles 4 with 1.7B
GRAD_ACCUM     = 8      # Effective batch = 32
LR             = 2e-4
WARMUP_RATIO   = 0.05

# --- Intents / Scopes (must match app/agent/router/config.py) ---
VALID_INTENTS = [
    "current_weather", "hourly_forecast", "daily_forecast", "weather_overview",
    "rain_query", "temperature_query", "wind_query", "humidity_fog_query",
    "historical_weather", "location_comparison", "activity_weather",
    "expert_weather_param", "weather_alert", "seasonal_context", "smalltalk_weather",
]
VALID_SCOPES = ["city", "district", "ward"]

# --- System prompt (must match ROUTER_SYSTEM_PROMPT in config.py) ---
SYSTEM_PROMPT = 'Phân loại intent và scope cho câu hỏi thời tiết Hà Nội. Trả về JSON.

## Intents:
- current_weather: thời tiết NGAY LÚC NÀY (nhiệt độ, trời nắng/mưa, chung chung)
- hourly_forecast: diễn biến CHI TIẾT THEO GIỜ trong ngày (không chỉ hỏi mưa)
- daily_forecast: dự báo NHIỀU NGÀY tới (3 ngày, tuần tới, cuối tuần)
- weather_overview: TỔNG QUAN, tóm tắt thời tiết hôm nay/ngày mai (không hỏi thông số cụ thể)
- rain_query: hỏi CÓ MƯA KHÔNG, mưa bao lâu, xác suất mưa, mưa lúc nào tạnh
- temperature_query: hỏi CỤ THỂ VỀ NHIỆT ĐỘ (bao nhiêu độ, nóng/lạnh thế nào)
- wind_query: hỏi CỤ THỂ VỀ GIÓ (gió mạnh không, hướng gió, tốc độ gió)
- humidity_fog_query: hỏi về ĐỘ ẨM, SƯƠNG MÙ, sương muối
- historical_weather: thời tiết NGÀY/TUẦN TRƯỚC, dữ liệu QUÁ KHỨ
- location_comparison: SO SÁNH thời tiết giữa các quận/phường/địa điểm
- activity_weather: thời tiết có PHÙ HỢP ĐỂ LÀM hoạt động X không (chạy bộ, picnic, du lịch)
- expert_weather_param: thông số KỸ THUẬT chuyên sâu (áp suất, tia UV, điểm sương, tầm nhìn)
- weather_alert: CẢNH BÁO, nguy hiểm, bão, ngập, thay đổi đột ngột
- seasonal_context: SO SÁNH với hôm qua/tuần trước, xu hướng, bất thường theo MÙA
- smalltalk_weather: chào hỏi, ngoài phạm vi (không phải Hà Nội), câu hỏi không liên quan thời tiết

## Scopes:
- city: toàn Hà Nội, hoặc KHÔNG NÓI RÕ địa điểm
- district: nhắc tên QUẬN/HUYỆN (Ba Đình, Cầu Giấy...) hoặc ĐỊA ĐIỂM NỔI TIẾNG thuộc quận (Hồ Gươm→Hoàn Kiếm, Lăng Bác→Ba Đình, Nội Bài→Sóc Sơn...)
- ward: nhắc tên PHƯỜNG/XÃ (Phường Dịch Vọng Hậu, Xã Tiên Dương...)

## Multi-task output (khi có context từ lượt trước):
Nếu được cung cấp context (location/intent từ lượt trước) VÀ câu hỏi hiện tại thiếu địa điểm hoặc dùng đại từ (ở đó, thế còn, còn ..., vậy ...), hãy điền thêm trường "rewritten_query" với câu hỏi đầy đủ ngữ cảnh.

## Output format:
Khi không có context hoặc câu hỏi đầy đủ:
{"intent": "...", "scope": "...", "confidence": 0.92}

Khi có context và cần rewrite:
{"intent": "...", "scope": "...", "confidence": 0.91, "rewritten_query": "Dự báo thời tiết ngày mai ở quận Cầu Giấy như thế nào?"}'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(GGUF_DIR).mkdir(parents=True, exist_ok=True)

print(f"Base model:      {BASE_MODEL}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Train file:      {TRAIN_FILE.name}")
print(f"Val file:        {VAL_FILE.name}")
print(f"Max seq length:  {MAX_SEQ_LENGTH}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Cell 4: Load base model (Qwen3-1.7B 4-bit QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # auto: bfloat16 on A100, float16 on T4/P100
    load_in_4bit=True,
)
print(f"Model: {type(model).__name__}")
print(f"Vocab: {tokenizer.vocab_size}")
print(f"Dtype: {next(model.parameters()).dtype}")

In [ ]:
# Cell 5: Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 6: Chat template
# Qwen3 uses ChatML (same format as Qwen2.5).
# enable_thinking=False: disable <think> blocks for fast JSON router output.
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

sample = [{"role": "system", "content": "Test"}, {"role": "user", "content": "Hello"}]
try:
    preview = tokenizer.apply_chat_template(
        sample, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
except TypeError:
    preview = tokenizer.apply_chat_template(sample, tokenize=False, add_generation_prompt=True)
print("Chat template preview:")
print(preview[:200])

In [ ]:
# Cell 7: Load and validate data
# multitask_train.jsonl and multitask_val.jsonl use unified input/output format.
# Generated by: python scripts/router/generate_rewrite_data.py --use-llm

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    print(f"  Line {i} error: {e}")
    return records

def validate_record(rec, idx):
    """Validate input/output format record."""
    out = rec.get("output", {})
    if isinstance(out, str):
        try: out = json.loads(out)
        except: return False
    return (out.get("intent") in VALID_INTENTS and
            out.get("scope")  in VALID_SCOPES  and
            bool(rec.get("input")))

# Load training data
print(f"Loading {TRAIN_FILE}...")
all_records = load_jsonl(TRAIN_FILE)
valid_records = [r for i, r in enumerate(all_records) if validate_record(r, i)]
multitask_count = sum(1 for r in valid_records if r.get("output", {}).get("rewritten_query"))
print(f"  Total: {len(valid_records)} / {len(all_records)} valid")
print(f"  Routing-only: {len(valid_records) - multitask_count}")
print(f"  With rewrite: {multitask_count}")

# Load validation data
print(f"Loading {VAL_FILE}...")
val_raw = load_jsonl(VAL_FILE)
val_records = [r for i, r in enumerate(val_raw) if validate_record(r, i)]
print(f"  Val: {len(val_records)}")

# Intent distribution
intent_counts = Counter()
for r in valid_records:
    out = r.get("output", {})
    if isinstance(out, str): out = json.loads(out)
    intent_counts[out.get("intent", "?")] += 1

print("
Intent distribution:")
for k, v in sorted(intent_counts.items(), key=lambda x: -x[1]):
    pct = v / len(valid_records) * 100
    print(f"  {k:<30} {v:>4} ({pct:4.1f}%)")

In [ ]:
# Cell 8: Format data for SFT (completion-only training)
# Loss is computed only on the assistant turn.

def format_record(rec):
    user_msg = str(rec.get("input", "")).strip()
    out = rec.get("output", {})
    if isinstance(out, str): out = json.loads(out)

    output_dict = {
        "intent":     out["intent"],
        "scope":      out["scope"],
        "confidence": round(float(out.get("confidence", 0.9)), 2),
    }
    rw = out.get("rewritten_query")
    if rw and str(rw).strip():
        output_dict["rewritten_query"] = str(rw).strip()

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": user_msg},
        {"role": "assistant", "content": json.dumps(output_dict, ensure_ascii=False)},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )


print("Formatting training data...")
train_texts = [format_record(r) for r in valid_records]
val_texts   = [format_record(r) for r in val_records]

print("\nSample:")
print("-" * 60)
print(train_texts[0])
print("-" * 60)

lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f"Lengths (first 200): min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")
over = sum(1 for l in lengths if l > MAX_SEQ_LENGTH)
print(f"Over limit: {over}/200")

In [ ]:
# Cell 9: HuggingFace datasets
train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset   = Dataset.from_dict({"text": val_texts})
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
# Cell 10: Data collator (completion-only)
# Loss computed only on assistant turn.
RESPONSE_TEMPLATE = "<|im_start|>assistant\n"
response_ids = tokenizer.encode(RESPONSE_TEMPLATE, add_special_tokens=False)
print(f"Response template IDs: {response_ids}")
print(f"Decoded: {tokenizer.decode(response_ids)!r}")

data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_ids,
    tokenizer=tokenizer,
)

In [ ]:
# Cell 11: Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    report_to="none",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
)
prec = "bf16" if training_args.bf16 else "fp16"
print(f"Precision: {prec} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Steps/epoch: {len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)}")

In [ ]:
# Cell 12: SFT Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=data_collator,
    args=training_args,
    packing=False,  # Required for completion-only collator
)
if torch.cuda.is_available():
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total_v  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM reserved: {reserved:.1f} GB / {total_v:.1f} GB")

In [ ]:
# Cell 13: TRAIN
print("=" * 60)
print(f"Model:  {BASE_MODEL}")
print(f"LoRA:   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:  {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Epochs: {EPOCHS}")
print("=" * 60)

train_result = trainer.train()

print(f"Train loss:  {train_result.training_loss:.4f}")
print(f"Runtime:     {train_result.metrics['train_runtime']:.0f}s")
print(f"Samples/s:   {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# Cell 14: Save LoRA adapter
adapter_dir = Path(OUTPUT_DIR) / "lora_adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Adapter: {adapter_dir}")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name} ({f.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Cell 15: Inference test
FastLanguageModel.for_inference(model)

TEST_CASES = [
    ("Routing",    "Bay gio thoi tiet Cau Giay the nao?"),
    ("Forecast",   "Cuoi tuan Ha Noi the nao?"),
    ("Multi-task", '[CONTEXT: {"location": "Cau Giay", "intent": "current_weather", "turn": 1}]\nCon ngay mai?'),
    ("Activity",   "Sang mai di chay bo duoc khong?"),
    ("Compare",    "Dong Da va Cau Giay noi nao mat hon?"),
]

print("Inference tests:")
all_ok = True
for desc, user_msg in TEST_CASES:
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_msg}]
    try:
        txt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        txt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(txt, return_tensors="pt").to(model.device)
    import torch as _t
    with _t.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=128,
            temperature=1.0, do_sample=False, use_cache=True,
        )
    resp = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    ok = False
    try:
        ok = json.loads(resp).get("intent") in VALID_INTENTS
    except Exception:
        pass
    if not ok:
        all_ok = False
    status = "OK" if ok else "FAIL"
    print(f"  [{status}] {desc}: {resp[:80]}")
print(f"All valid: {all_ok}")

In [ ]:
# Cell 16: Export GGUF
# q8_0   = best quality (~1.8 GB) - recommended for Ollama production
# q4_k_m = smaller    (~1.1 GB) - fallback if RAM is constrained

print("Exporting GGUF q8_0...")
model.save_pretrained_gguf(GGUF_DIR + "/q8_0",   tokenizer, quantization_method="q8_0")

print("Exporting GGUF q4_k_m...")
model.save_pretrained_gguf(GGUF_DIR + "/q4_k_m", tokenizer, quantization_method="q4_k_m")

print("\nGGUF files:")
for f in sorted(Path(GGUF_DIR).rglob("*.gguf")):
    print(f"  {f.relative_to(GGUF_DIR)}  ({f.stat().st_size/1e9:.2f} GB)")

In [ ]:
# Cell 17: Generate Ollama Modelfile
modelfile = f"""FROM ./Qwen3-1.7B-router-q8_0.gguf

SYSTEM \"{SYSTEM_PROMPT}\"

PARAMETER temperature 0
PARAMETER num_predict 128
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
"""

mf = Path(GGUF_DIR) / "Modelfile"
mf.write_text(modelfile, encoding="utf-8")
print(f"Modelfile: {mf}")
print("\nDeploy on Ollama:")
print("  1. Download q8_0 GGUF from Kaggle Output tab")
print("  2. Rename to: Qwen3-1.7B-router-q8_0.gguf")
print("  3. ollama create hanoi-weather-router -f Modelfile")
print("  4. Test: ollama run hanoi-weather-router")
print("\n.env: OLLAMA_MODEL=hanoi-weather-router  USE_SLM_ROUTER=true")

In [ ]:
# Cell 18: (Optional) Push to HuggingFace Hub
PUSH_TO_HUB  = False
HUB_MODEL_ID = "your-username/qwen3-1.7b-hanoi-weather-router"

if PUSH_TO_HUB:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = os.getenv("HF_TOKEN", "")
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        model.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        tokenizer.push_to_hub(HUB_MODEL_ID, token=hf_token, private=True)
        print(f"Pushed: https://huggingface.co/{HUB_MODEL_ID}")
    else:
        print("HF_TOKEN not in Kaggle Secrets")
else:
    print("Hub push skipped (set PUSH_TO_HUB=True to enable)")

In [ ]:
# Cell 19: Summary
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Base:   {BASE_MODEL}")
print(f"LoRA:   r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Train:  {len(train_dataset)} ({multitask_count} with-rewrite)")
print(f"Loss:   {train_result.training_loss:.4f}")
print()
print("Output files (Kaggle Output tab):")
for root, dirs, files in os.walk("/kaggle/working"):
    for fname in sorted(files):
        fp = os.path.join(root, fname)
        sz = os.path.getsize(fp) / 1e6
        if sz > 0.1:
            rel = os.path.relpath(fp, "/kaggle/working")
            print(f"  {rel:<55} {sz:7.1f} MB")